# Fine-Tuning Google MuRIL for Multilingual Intent Classification
### CampusMate Minor Project - Feature 03
This Google Colab notebook fine-tunes `google/muril-base-cased` (or `ai4bharat/IndicBERTv2-MLM-only`) on `train.csv` and evaluates on `val.csv` and `test.csv` across 20 college intents in English, Hindi, Gujarati, Hinglish, and Gujlish.

In [ ]:
!pip install -q transformers datasets evaluate accelerate torch scikit-learn

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load datasets
df_train = pd.read_csv('train.csv')
df_val = pd.read_csv('val.csv')
df_test = pd.read_csv('test.csv')

intents = sorted(df_train['intent'].unique())
label2id = {label: i for i, label in enumerate(intents)}
id2label = {i: label for i, label in enumerate(intents)}

df_train['label'] = df_train['intent'].map(label2id)
df_val['label'] = df_val['intent'].map(label2id)
df_test['label'] = df_test['intent'].map(label2id)

MODEL_NAME = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_ds = Dataset.from_pandas(df_train).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(df_val).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(df_test).map(tokenize_fn, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(intents),
    id2label=id2label,
    label2id=label2id
)
model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results_muril',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    seed=42
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = np.mean(preds == labels)
    return {'accuracy': acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

In [ ]:
# Evaluate on held-out test split
test_results = trainer.evaluate(test_ds)
print("Held-out Test Evaluation Results:", test_results)